# 03_gold: Build the star schema from Silver

**Input:** 6 Silver Delta tables in `silver/<table>`
**Output:** 5 Gold Delta tables in `gold/` (overwrite → idempotent)
- Dimensions: `dim_customers`, `dim_products`, `dim_date`
- Facts: `fact_orders` (grain: one row per order), `fact_events` (grain: one row per website action)

**Design:** galaxy schema (2 facts sharing conformed dimensions); dimensions built before facts; every fact key validated against its dimension before writing

In [0]:
from pyspark.sql import functions as F

SILVER_PATH = "abfss://silver@ecommercedatalakebp01.dfs.core.windows.net"
GOLD_PATH = "abfss://gold@ecommercedatalakebp01.dfs.core.windows.net"

In [0]:
# Quality gate: fail the run if any fact row has a key missing from its dimension
def assert_no_orphans(fact, dim, fact_key, dim_key, label):
    orphans = fact.join(dim, fact[fact_key] == dim[dim_key], "left_anti").count()
    assert orphans == 0, f"{label}: {orphans} rows with {fact_key} not found in dimension"

In [0]:
df_returns = spark.read.format("delta").load(f"{SILVER_PATH}/returns")

print("total returns:", df_returns.count())
print("distinct order_id:", df_returns.select("order_id").distinct().count())

## Dimensions
### dim_customers
- Silver customers (business columns only) + 2 unknown-member rows: `UNKNOWN` (orders with no customer) and `GUEST` (events from visitors not logged in)
- Special rows use "Unknown" placeholders and a null `signup_date` (no invented data)

In [0]:
df_customers = (
    spark.read.format("delta").load(f"{SILVER_PATH}/customers")
    .select("customer_id", "customer_name", "email", "city", "state", "country", "signup_date")
)

special_rows = [
    ("UNKNOWN", "Unknown customer", "Unknown", "Unknown", "Unknown", "India", None),
    ("GUEST",   "Guest visitor",    "Unknown", "Unknown", "Unknown", "India", None),
]
df_special = spark.createDataFrame(special_rows, schema=df_customers.schema)

dim_customers = df_customers.unionByName(df_special).withColumn("_processed_at", F.current_timestamp())

In [0]:
assert dim_customers.select("customer_id").distinct().count() == dim_customers.count(),"Duplicates have been found"

In [0]:
dim_customers=dim_customers.withColumn("_processed_at", F.current_timestamp())

In [0]:
dim_customers.write.mode("overwrite").format("delta").save(f"{GOLD_PATH}/dim_customers")

In [0]:
check=spark.read.format("delta").load(f"{GOLD_PATH}/dim_customers")
print("rows:",check.count())
print("columns:", check.columns)

### dim_products
- Business columns only (`price_sign_corrected` and tracking columns stay in Silver)

In [0]:
dim_products = (
    spark.read.format("delta").load(f"{SILVER_PATH}/products")
    .select("product_id", "product_name", "category", "price", "supplier_id")
    .withColumn("_processed_at", F.current_timestamp())
)

In [0]:
assert dim_products.select("product_id").distinct().count() == dim_products.count(),"Duplicates have been found"


In [0]:
dim_products.write.mode("overwrite").format("delta").save(f"{GOLD_PATH}/dim_products")

In [0]:
check = spark.read.format("delta").load(f"{GOLD_PATH}/dim_products")
print("rows:", check.count())
print("columns:", check.columns)

### dim_date
- Generated calendar 2023-01-01 → 2026-12-31: covers every fact date; no gaps for days without orders
- `year_month` keeps e.g. March 2025 and March 2026 separate in monthly trends

In [0]:
dim_date = (
    spark.range(1)
    .select(F.explode(F.sequence(F.to_date(F.lit("2023-01-01")), F.to_date(F.lit("2026-12-31")))).alias("date"))
    .withColumn("year",        F.year("date"))
    .withColumn("quarter",     F.quarter("date"))
    .withColumn("month",       F.month("date"))
    .withColumn("month_name",  F.date_format("date", "MMMM"))
    .withColumn("year_month",  F.date_format("date", "yyyy-MM"))
    .withColumn("day",         F.dayofmonth("date"))
    .withColumn("day_name",    F.date_format("date", "EEEE"))
    .withColumn("is_weekend",  F.dayofweek("date").isin(1, 7))   # 1 = Sunday, 7 = Saturday
    .withColumn("_processed_at", F.current_timestamp())
)

In [0]:
assert dim_date.select("date").distinct().count() == dim_date.count(), "Duplicate dates found!"
dim_date.write.mode("overwrite").format("delta").save(f"{GOLD_PATH}/dim_date")
check = spark.read.format("delta").load(f"{GOLD_PATH}/dim_date")
print("rows:", check.count())       
print("columns:", check.columns)     

In [0]:
df_orders = (spark.read.format("delta").load(f"{SILVER_PATH}/orders")
    .select("order_id", "customer_id", "product_id", "order_date", "quantity", "price", "revenue", "order_status"))
df_payments = (spark.read.format("delta").load(f"{SILVER_PATH}/payments")
    .select("order_id", "payment_method", "payment_status", "payment_amount"))
df_returns = (spark.read.format("delta").load(f"{SILVER_PATH}/returns")
    .select("order_id", "return_date", "return_reason", "refund_amount"))

# joins below are only safe if each order has at most one payment and one return
assert df_payments.select("order_id").distinct().count() == df_payments.count(), "payments: more than one payment per order"
assert df_returns.select("order_id").distinct().count() == df_returns.count(), "returns: more than one return per order"

## Facts
### fact_orders
- Grain: one row per order (50,000)
- Orders LEFT JOIN payments (exactly 1 per order) LEFT JOIN returns (at most 1 per order): left joins keep every order
- `is_returned` = order has a return; net revenue in reports = `payment_status == "Success"`
- Validated: grain unchanged after joins; every key exists in its dimension

In [0]:
fact_orders = (
    df_orders
    .join(df_payments, on="order_id", how="left")
    .join(df_returns, on="order_id", how="left")
    .withColumn("is_returned", F.col("return_date").isNotNull())
    .withColumn("_processed_at", F.current_timestamp())
)

In [0]:
dim_customers = spark.read.format("delta").load(f"{GOLD_PATH}/dim_customers")
dim_products  = spark.read.format("delta").load(f"{GOLD_PATH}/dim_products")
dim_date      = spark.read.format("delta").load(f"{GOLD_PATH}/dim_date")

assert fact_orders.count() == df_orders.count(), "fact_orders: joins changed the row count (grain broken)"
assert fact_orders.select("order_id").distinct().count() == fact_orders.count(), "fact_orders: duplicate order_id"
assert_no_orphans(fact_orders, dim_customers, "customer_id", "customer_id", "fact_orders")
assert_no_orphans(fact_orders, dim_products,  "product_id",  "product_id",  "fact_orders")
assert_no_orphans(fact_orders, dim_date,      "order_date",  "date",        "fact_orders")

In [0]:
fact_orders.write.mode("overwrite").format("delta").save(f"{GOLD_PATH}/fact_orders")
check = spark.read.format("delta").load(f"{GOLD_PATH}/fact_orders")
print("rows:", check.count())       
print("columns:", check.columns)

### fact_events
- Grain: one row per website action (clickstream), 149,998 rows
- `unknown_event` rows filtered out here (kept in Silver as the faithful record)
- `event_date` = date part of `event_timestamp`, so it joins to `dim_date`
- Validated: every customer (incl. GUEST), product and date exists in its dimension

In [0]:
df_events = spark.read.format("delta").load(f"{SILVER_PATH}/website_events")    
fact_events=(df_events
             .filter(F.col("event_type")!="unknown_event")
             .select("event_id","customer_id","product_id","event_type","event_timestamp")
             .withColumn("event_date", F.to_date(F.col("event_timestamp")))
             .withColumn("_processed_at", F.current_timestamp()))

In [0]:
assert fact_events.select("event_id").distinct().count() == fact_events.count(), "fact_events: duplicate event_id"
assert fact_events.filter(F.col("event_type") == "unknown_event").count() == 0, "fact_events: unknown_event not filtered"
assert_no_orphans(fact_events, dim_customers, "customer_id", "customer_id", "fact_events")
assert_no_orphans(fact_events, dim_products,  "product_id",  "product_id",  "fact_events")
assert_no_orphans(fact_events, dim_date,      "event_date",  "date",        "fact_events")

In [0]:
fact_events.write.mode("overwrite").format("delta").save(f"{GOLD_PATH}/fact_events")

check = spark.read.format("delta").load(f"{GOLD_PATH}/fact_events")
print("rows:", check.count())
print("columns:", check.columns)